# AI Programming Foundations Project
### Author: Patryk Pilarski
### Dataset: Titanic - Machine Learning from Disaster

This project presents a complete, reproducible data workflow using Python. In the following cells Titanic dataset will be loaded, cleaned, explored and visualized. Findings can be found at the end of the file.

#### Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#### Load data

In [2]:
df = pd.read_csv("train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


#### Check for issues in data

In [3]:
# missing values in columns Age, Cabin and Embarked
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [4]:
# Encyclopedia Titanica states that they embarked in Southampton
# https://www.encyclopedia-titanica.org/titanic-survivor/martha-evelyn-stone.html
df.loc[df['Embarked'].isna(),:]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN


In [5]:
# multiple passengers under one ticket
df["Ticket"].value_counts().sort_values(ascending=False).head(10)

Ticket
347082          7
CA. 2343        7
1601            7
3101295         6
CA 2144         6
347088          6
382652          5
S.O.C. 14879    5
LINE            4
2666            4
Name: count, dtype: int64

In [6]:
# mostly missing values and high cardinality of cabin numbers
print(f"percent of missing values: {df["Cabin"].isna().sum() / df.shape[0]:.2f}\nnumber of unique values: {len(pd.unique(df["Cabin"]))}")

percent of missing values: 0.77
number of unique values: 148


In [7]:
# first letters of cabin numbers correspond to ship decks https://www.encyclopedia-titanica.org/titanic-deckplans/
df["Cabin"].str[:1].value_counts()

Cabin
C    59
B    47
D    33
E    32
A    15
F    13
G     4
T     1
Name: count, dtype: int64

In [8]:
# all names are unique
print(f"dataset size: {df.shape[0]}\nnumber of unique names {len(pd.unique(df["Name"]))}")

dataset size: 891
number of unique names 891


In [9]:
# limited number titles in names (need some cleaning)
df["Name"].str.extract(r" ([A-Za-z]+)\.").value_counts()

0       
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

In [10]:
# with many missing values for age, without more sophisticated methods,
# imputing median values from subgroups seems like the best approach
# (removing almost 200 rows from 900 rows dataset seems too drastic)
df.groupby(["Pclass", "Sex"])["Age"].describe()

count       mean        std   min     25%   50%    75%   max
Pclass Sex                                                                 
1      female   85.0  34.611765  13.612052  2.00  23.000  35.0  44.00  63.0
       male    101.0  41.281386  15.139570  0.92  30.000  40.0  51.00  80.0
2      female   74.0  28.722973  12.872702  2.00  22.250  28.0  36.00  57.0
       male     99.0  30.740707  14.793894  0.67  23.000  30.0  36.75  70.0
3      female  102.0  21.750000  12.729964  0.75  14.125  21.5  29.75  63.0
       male    253.0  26.507589  12.159514  0.42  20.000  25.0  33.00  74.0

#### Data cleaning functions

In [11]:
def transform_embarked(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform Embarked column. Fill missing values for passengers
    with ticket '113572' and map codes to port names.
    """
    df_ = df.copy()
    df_.loc[df["Ticket"] == "113572", "Embarked"] = "S"
    df_["Embarked"] = df_["Embarked"].map({"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"})
    return df_

In [12]:
def replace_cabin_with_deck(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create Deck column from values extracted from cabin number,
    fill missing values for deck with 'X' and remove Cabin column.
    """
    df_ = df.copy()
    df_["Deck"] = df_["Cabin"].str[:1]
    df_["Deck"] = df_["Deck"].fillna("X")
    df_.drop(columns=["Cabin"], inplace=True)
    return df_

In [13]:
def add_companions_count(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create Companions column based on the number of people under the same ticket.
    In some cases passengers traveled with people other than faimily
    """
    df_ = df.copy()
    df_["Companions"] = df_.groupby("Ticket")["Ticket"].transform("count") - 1
    return df_

In [14]:
def extract_title(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create column Title with values extracted from names. Reduce title cardinality
    by remapping rare cases to more common classes.
    """
    df_ = df.copy()
    title_map = {
        "Mlle": "Miss", "Ms": "Mrs", "Mme": "Mrs", "Lady": "Noble", "Countess": "Noble",
        "Don": "Noble", "Dona": "Noble", "Jonkheer": "Noble", "Sir": "Noble", "Rev": "Noble",
        "Major": "Noble", "Col": "Noble", "Capt": "Noble", "Dr": "Noble"
    }
    df_["Title"] = df_["Name"].str.extract(r" ([A-Za-z]+)\.")
    df_["Title"] = df_["Title"].replace(title_map)
    return df_

In [15]:
def fill_missing_age(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """
    Fill missing values in Age column with median values from groups defined by group_cols.
    """
    df_ = df.copy()
    df_["tmp"] = df_.groupby(group_cols)["Age"].transform("median")
    df_["Age"] = df_["Age"].fillna(df_["tmp"])
    df_.drop(columns=["tmp"], inplace=True)
    return df_

#### Clean data

In [16]:
df_clean = transform_embarked(df)
df_clean = replace_cabin_with_deck(df_clean)
df_clean = add_companions_count(df_clean)
df_clean = extract_title(df_clean)
df_clean = fill_missing_age(df_clean, ["Pclass", "Sex", "Deck", "Title"])
df_clean.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Deck,Companions,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,Southampton,X,0,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,Cherbourg,C,0,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,Southampton,X,0,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,Southampton,C,1,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,Southampton,X,0,Mr
